# Agent 3 — Program Agent

**What Agent 3 does:** within each university Agent 2 has already shortlisted, Agent 3 identifies which specific program or specialization best fits the student's stated academic interests -- a distinction Agent 2 doesn't make on its own, since Agent 2 matches at the university/target-degree level.

**Input:**
- `state["matched_universities"]` -- Agent 2's shortlisted universities, each carrying its candidate programs
- `state["profile"]` -- including `interests_text`, a free-text description of the student's academic interests, richer than the single `target_program` string Agent 2 uses

**Processing:**
1. Parse each program's specialization from its name or department field (Section 2) -- validated directly against the real catalog, not assumed correct.
2. Look up duration and delivery mode from the knowledge base (`agent2_final_2tier.csv` -- the same knowledge base Agent 2 uses; no separate dataset needed).
3. Group every program in the knowledge base into specialization clusters via KMeans, run over embeddings of each program's curriculum/faculty-research text (Section 4).
4. Score the semantic fit between the student's interests and each candidate program in two stages: bi-encoder cosine similarity for fast retrieval-quality scoring, then a CrossEncoder rerank for a more accurate final ranking -- the same two-stage pattern Agent 2 already uses for university retrieval (Section 5).
5. Blend the CrossEncoder-reranked score with how well the interests match the program's specialization cluster as a whole, so the fit score isn't swung entirely by the wording of one program's description field.
6. Combine the specialization-fit score with Agent 2's `match_score`/`admit_probability` into a single ranked, per-program list.

**Output:** `state["programs"]` -- a flat, ranked list of `{university, program_name, specialization, duration, mode, specialization_fit_score, match_score, admit_probability, tier}`.

**Connected agents:** reads Agent 2's (University Matching) shortlist; its output is read by Agent 4 (Financial) and Agent 6 (Career).

## 1. Setup

### Colab environment fix (run this first, then restart runtime)
Same `PIL._typing._Ink` conflict as Agents 2 and 7 -- see those notebooks for details.

**Fix:** run the cell below once, then **Runtime -> Restart session**, then re-run from the top.

In [ ]:
!pip install -U --force-reinstall pillow

In [1]:
!pip install -q -U pillow torchvision transformers sentence-transformers
print("Upgraded. Now go to Runtime -> Restart session, then re-run all cells from the top.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 995.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 M

In [2]:
!pip install qdrant-client sentence-transformers scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 5.0 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import re

from google.colab import files
print("Upload agent2_final_2tier.csv -- the same knowledge base Agent 2 uses (no separate dataset needed)")
uploaded = files.upload()

Upload agent2_final_2tier.csv -- the same knowledge base Agent 2 uses (no separate dataset needed)


Saving agent2_final_2tier.csv to agent2_final_2tier (2).csv


In [5]:
df = pd.read_csv("agent2_final_2tier.csv")
df = df.fillna("")
print(df.shape)
print(df.columns.tolist())

(4500, 44)
['university_id', 'university_name', 'state', 'city', 'latitude', 'longitude', 'tuition_in_state_usd', 'tuition_out_state_usd', 'university_admission_rate', 'student_size', 'us_news_ranking', 'world_ranking', 'campus_setting', 'website_url', 'avg_cost_of_living_monthly', 'international_student_pct', 'public_or_private', 'program_id', 'program_name', 'degree_type', 'department', 'min_gpa', 'min_gre_quant', 'min_gre_verbal', 'min_gmat', 'min_toefl', 'min_ielts', 'application_deadline_fall', 'application_fee_usd', 'duration_months', 'stem_designated', 'program_admit_rate', 'description_text', 'curriculum_highlights', 'faculty_research_areas', 'avg_starting_salary_usd', 'male_acceptance_rate', 'female_acceptance_rate', 'historical_admit_count', 'historical_admit_rate', 'data_quality', 'track', 'tier_score', 'tier']


In [6]:
from sentence_transformers import SentenceTransformer

try:
    embed_model
    print("Reusing existing embed_model from an earlier agent in this session.")
except NameError:
    embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    print("Loaded a fresh embed_model.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded a fresh embed_model.


## 2. Specialization extraction -- parse from program name, validated against the real catalog
Program names mostly follow `"<Degree> <Department> - <Specialization>"` (e.g. `"MS Computer Science - Systems"`), but checking the real catalog directly (not assuming) finds 18 of 90 unique program names (20%) have **no** `" - "` delimiter -- non-specialized "general track" programs like `"MS Computer Science"` or `"MS Data Science"`. The original parser fell back to the row's `department` for these, which collapses genuinely different programs into the same label: `"MS Computer Science"`, `"MS Data Science"`, and `"MS Artificial Intelligence"` all land on `"Computer & Information Sciences"`.

The fix: for a program with no `" - "` delimiter, strip the leading degree token (`MS`/`MEng`/`MBA`/...) and use what's left -- the program's own field name is the real distinguishing label here, not its (much coarser) department.

In [7]:
def extract_specialization(program_name: str, department: str = "") -> str:
    """Pulls the specialization out of a program name if present. If there's no
    ' - ' delimiter, strips the leading degree token and uses the remaining field
    name (e.g. 'MS Computer Science' -> 'Computer Science') rather than falling back
    to department, which is too coarse to distinguish different general-track programs."""
    if " - " in program_name:
        return program_name.split(" - ", 1)[1].strip()

    tokens = program_name.split(" ", 1)
    if len(tokens) == 2 and tokens[0].isupper() and len(tokens[0]) <= 5:
        remainder = tokens[1].strip()
        if remainder:
            return remainder
    return department if department else "General"


df["specialization"] = df.apply(
    lambda row: extract_specialization(row["program_name"], row.get("department", "")), axis=1
)
print(df[["program_name", "specialization"]].drop_duplicates().head(10))

                                 program_name           specialization
0                 MBA Business Administration  Business Administration
1       MBA Business Administration - Finance                  Finance
2     MBA Business Administration - Marketing                Marketing
3    MBA Business Administration - Operations               Operations
4    MBA Business Administration - Leadership               Leadership
250                       MS Computer Science         Computer Science
251                           MS Data Science             Data Science
252                MS Artificial Intelligence  Artificial Intelligence
253                          MS Cybersecurity            Cybersecurity
254                    MS Information Systems      Information Systems


### Validate the fix against the real catalog
Measures the actual collision rate -- how many distinct programs get squashed into the same specialization label -- before and after, rather than assuming the fix helped.

In [8]:
def extract_specialization_old(program_name, department=""):
    if " - " in program_name:
        return program_name.split(" - ", 1)[1].strip()
    return department if department else "General"


unique_names = df["program_name"].drop_duplicates()
has_dash = unique_names.str.contains(" - ")
no_dash = unique_names[~has_dash]

old_labels = {n: extract_specialization_old(n, df[df.program_name == n].iloc[0].department) for n in no_dash}
new_labels = {n: extract_specialization(n, df[df.program_name == n].iloc[0].department) for n in no_dash}

print(f"Programs with no ' - ' delimiter: {len(no_dash)} of {len(unique_names)} unique names ({len(no_dash)/len(unique_names):.0%})")
print(f"OLD parser: {len(no_dash)} distinct programs collapse into {len(set(old_labels.values()))} distinct specialization labels")
print(f"NEW parser: {len(no_dash)} distinct programs map to {len(set(new_labels.values()))} distinct specialization labels")
print()
print("Sample:")
for n in list(no_dash)[:6]:
    print(f"  '{n}':  '{old_labels[n]}'  ->  '{new_labels[n]}'")

Programs with no ' - ' delimiter: 18 of 90 unique names (20%)
OLD parser: 18 distinct programs collapse into 4 distinct specialization labels
NEW parser: 18 distinct programs map to 18 distinct specialization labels

Sample:
  'MBA Business Administration':  'Business, Management & Marketing'  ->  'Business Administration'
  'MS Computer Science':  'Computer & Information Sciences'  ->  'Computer Science'
  'MS Data Science':  'Computer & Information Sciences'  ->  'Data Science'
  'MS Artificial Intelligence':  'Computer & Information Sciences'  ->  'Artificial Intelligence'
  'MS Cybersecurity':  'Computer & Information Sciences'  ->  'Cybersecurity'
  'MS Information Systems':  'Computer & Information Sciences'  ->  'Information Systems'


## 3. Duration + mode
`duration_months` already exists in the dataset. `mode` (on-campus / online / hybrid) isn't in this CSV -- defaults to `"On-campus"` with a flag so you know it's a placeholder, not scraped data. If your team can source this per-program (most course catalogs list it), swap the default for a real lookup.

In [9]:
DEFAULT_MODE = "On-campus"  # placeholder -- this dataset does not carry a delivery-mode column

def get_duration_and_mode(row) -> dict:
    duration_months = row.get("duration_months", None)
    duration_label = f"{int(duration_months)} months" if duration_months not in (None, "", 0) else "Not specified"
    mode = row.get("delivery_mode", DEFAULT_MODE)  # falls back to default if column absent
    return {"duration": duration_label, "mode": mode, "mode_is_placeholder": "delivery_mode" not in row.index}

## 4. Specialization clustering (KMeans)
Groups every program in the knowledge base into specialization clusters by embedding each program's curriculum-highlights/faculty-research text and running KMeans over the resulting vectors, with the cluster count chosen by silhouette score. Used in Section 6 to smooth the specialization-fit score: a student's interests are compared both against the specific program's text and against its cluster's overall theme, so a thin or unevenly-written description field doesn't swing the score as much as the specialization's broader direction does.

In [10]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def build_program_cluster_text(row):
    return f"{row.get('curriculum_highlights', '')} {row.get('faculty_research_areas', '')}".strip()

program_cluster_texts = df.apply(build_program_cluster_text, axis=1)
program_vectors = embed_model.encode(program_cluster_texts.tolist(), show_progress_bar=True)

best_k, best_silhouette = None, -1
for k in range(4, 13):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(program_vectors)
    score = silhouette_score(program_vectors, labels)
    if score > best_silhouette:
        best_k, best_silhouette = k, score

print(f"Selected k={best_k} specialization clusters (silhouette={best_silhouette:.3f})")

specialization_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(program_vectors)
df["specialization_cluster"] = specialization_kmeans.labels_
cluster_centroids = specialization_kmeans.cluster_centers_

for cluster_id in range(best_k):
    top_specs = df.loc[df["specialization_cluster"] == cluster_id, "specialization"].value_counts().head(3)
    print(f"Cluster {cluster_id} ({(df['specialization_cluster'] == cluster_id).sum()} programs): "
          f"{', '.join(top_specs.index.tolist())}")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Selected k=12 specialization clusters (silhouette=0.457)
Cluster 0 (250 programs): Industrial Engineering, Optimization, Supply Chain
Cluster 1 (500 programs): Data Science, Computer Science, Systems
Cluster 2 (250 programs): Software Engineering, DevOps, Testing
Cluster 3 (500 programs): Information Systems, Business Analytics, Cloud Computing
Cluster 4 (500 programs): Mechanical Engineering, Chemical Engineering, Thermal Sciences
Cluster 5 (500 programs): Artificial Intelligence, Robotics, NLP
Cluster 6 (500 programs): VLSI, Embedded Systems, Engineering
Cluster 7 (500 programs): Statistics, Applied Mathematics, Statistical Inference
Cluster 8 (250 programs): Civil Engineering, Transportation, Geotechnical Engineering
Cluster 9 (250 programs): Biomedical Engineering, Biomechanics, Biomaterials
Cluster 10 (250 programs): Cybersecurity, Cryptography, Cloud Security
Cluster 11 (250 programs): Business Administration, Finance, Marketing


## 5. Specialization fit scoring -- two-stage retrieval + rerank
The original version scored fit with a single bi-encoder cosine similarity. Bi-encoders embed the student's interests and the program's text independently, so they're fast but miss cross-term interactions between the two texts. A CrossEncoder scores the pair jointly instead -- slower, so it's not used for the initial Agent-2-style retrieval over thousands of programs, but Agent 3 only reranks a handful of already-shortlisted programs per university, exactly the regime a CrossEncoder rerank is meant for. This is the same two-stage retrieve-then-rerank pattern Agent 2 already uses for university matching.

In [11]:
from sentence_transformers import CrossEncoder

try:
    specialization_reranker
    print("Reusing existing specialization_reranker from an earlier agent in this session.")
except NameError:
    specialization_reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    print("Loaded a fresh CrossEncoder reranker.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded a fresh CrossEncoder reranker.


In [12]:
def compute_specialization_fit(student_interests: str, curriculum_highlights: str, faculty_research_areas: str,
                                cluster_id=None) -> float:
    """Two-stage fit score: bi-encoder cosine similarity (fast, used as the fallback and
    for the cluster-centroid comparison) reranked by a CrossEncoder (slower, but accurate
    for the small per-university candidate set Agent 3 actually scores), then blended
    with similarity to the program's specialization cluster centroid when known."""
    if not student_interests:
        return 0.5

    program_text = f"{curriculum_highlights} {faculty_research_areas}".strip()
    if not program_text:
        return 0.5

    interest_vec = embed_model.encode(student_interests)
    program_vec = embed_model.encode(program_text)
    cos_sim = float(np.dot(interest_vec, program_vec) / (np.linalg.norm(interest_vec) * np.linalg.norm(program_vec)))
    bi_encoder_fit = round((cos_sim + 1) / 2, 3)

    cross_encoder_raw = specialization_reranker.predict([(student_interests, program_text)])[0]
    cross_encoder_fit = float(1 / (1 + np.exp(-cross_encoder_raw)))

    reranked_fit = round(0.7 * cross_encoder_fit + 0.3 * bi_encoder_fit, 3)

    if cluster_id is None or cluster_id not in range(len(cluster_centroids)):
        return reranked_fit

    centroid = cluster_centroids[cluster_id]
    cluster_cos_sim = float(np.dot(interest_vec, centroid) / (np.linalg.norm(interest_vec) * np.linalg.norm(centroid)))
    cluster_fit = round((cluster_cos_sim + 1) / 2, 3)

    return round(0.8 * reranked_fit + 0.2 * cluster_fit, 3)

## 6. Validate the fit scoring against a small hand-labeled set
Ranking quality doesn't have a ground-truth dataset the way Agent 1/2's admit predictions do, so this builds a small, honestly-constructed test: for each of several student interest statements, a plausible best-matching specialization and a plausible poor-matching one, drawn from the real catalog's own curriculum/research text. This is a top-1 sanity check, not a claim of rigorous statistical validation -- useful for catching a method that's obviously broken, not for reporting a precise accuracy figure.

In [13]:
validation_cases = [
    {
        "interests": "distributed systems, cloud infrastructure, and large-scale backend engineering",
        "expected_better": "Systems", "expected_worse": "Regression",
    },
    {
        "interests": "reinforcement learning and training autonomous agents to make sequential decisions",
        "expected_better": "Reinforcement Learning", "expected_worse": "Water Resources",
    },
    {
        "interests": "financial modeling, marketing analytics, and using data to guide business strategy",
        "expected_better": "Marketing Analytics", "expected_worse": "VLSI",
    },
    {
        "interests": "designing control systems and planning algorithms for autonomous robots",
        "expected_better": "Control", "expected_worse": "Enterprise Systems",
    },
    {
        "interests": "semiconductor and chip design, especially digital circuit layout",
        "expected_better": "VLSI", "expected_worse": "Bioinstrumentation",
    },
]

def score_methods(interests, spec_label):
    rows = df[df["specialization"] == spec_label]
    if rows.empty:
        return None
    row = rows.iloc[0]
    curriculum, research = row.get("curriculum_highlights", ""), row.get("faculty_research_areas", "")
    cluster_id = row.get("specialization_cluster")

    interest_vec = embed_model.encode(interests)
    program_vec = embed_model.encode(f"{curriculum} {research}".strip())
    cos_sim = float(np.dot(interest_vec, program_vec) / (np.linalg.norm(interest_vec) * np.linalg.norm(program_vec)))
    bi_encoder_only = round((cos_sim + 1) / 2, 3)

    full_score = compute_specialization_fit(interests, curriculum, research, cluster_id=cluster_id)
    return bi_encoder_only, full_score


results = {"bi_encoder_only": 0, "full_pipeline": 0}
total = 0
for case in validation_cases:
    better = score_methods(case["interests"], case["expected_better"])
    worse = score_methods(case["interests"], case["expected_worse"])
    if better is None or worse is None:
        continue
    total += 1
    print(f"\nInterests: {case['interests'][:60]}...")
    print(f"  {case['expected_better']:<24} bi_encoder={better[0]:.3f}  full_pipeline={better[1]:.3f}")
    print(f"  {case['expected_worse']:<24} bi_encoder={worse[0]:.3f}  full_pipeline={worse[1]:.3f}")
    if better[0] > worse[0]:
        results["bi_encoder_only"] += 1
    if better[1] > worse[1]:
        results["full_pipeline"] += 1

print(f"\n=== Top-1 sanity check: correctly ranked the better match higher, out of {total} cases ===")
print(f"Bi-encoder only (original):        {results['bi_encoder_only']}/{total}")
print(f"Full pipeline (rerank + cluster):   {results['full_pipeline']}/{total}")
print("\nRead the printed scores above, not just the pass/fail count -- with only "
      f"{total} hand-built cases this is a sanity check, not a statistically powered evaluation.")


Interests: distributed systems, cloud infrastructure, and large-scale b...
  Systems                  bi_encoder=0.850  full_pipeline=0.381
  Regression               bi_encoder=0.775  full_pipeline=0.351

Interests: reinforcement learning and training autonomous agents to mak...
  Reinforcement Learning   bi_encoder=0.868  full_pipeline=0.390
  Water Resources          bi_encoder=0.764  full_pipeline=0.335

Interests: financial modeling, marketing analytics, and using data to g...
  Marketing Analytics      bi_encoder=0.912  full_pipeline=0.783
  VLSI                     bi_encoder=0.762  full_pipeline=0.340

Interests: designing control systems and planning algorithms for autono...
  Control                  bi_encoder=0.918  full_pipeline=0.646
  Enterprise Systems       bi_encoder=0.775  full_pipeline=0.346

Interests: semiconductor and chip design, especially digital circuit la...
  VLSI                     bi_encoder=0.886  full_pipeline=0.391
  Bioinstrumentation       bi_encod

## 7. Full agent function
Combines Agent 2's `match_score`/`admit_probability` (already computed per program) with this agent's `specialization_fit_score` into a single `combined_score`, then flattens Agent 2's university-grouped output into the framework's expected per-program shape.

In [14]:
def program_agent(state: dict) -> dict:
    profile = state["profile"]
    student_interests = profile.get("interests_text", "")

    university_matches = state.get("matched_universities", [])
    if not university_matches:
        print("Warning: no matched_universities in state -- run Agent 2 first.")
        state["programs"] = []
        return state

    df_lookup = df.set_index(["university_name", "program_name"])

    flat_programs = []
    for uni in university_matches:
        for prog in uni["programs"]:
            key = (prog["university_name"], prog["program_name"])
            if key not in df_lookup.index:
                continue
            row = df_lookup.loc[key]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]

            specialization = extract_specialization(prog["program_name"], row.get("department", ""))
            duration_mode = get_duration_and_mode(row)
            fit_score = compute_specialization_fit(
                student_interests, row.get("curriculum_highlights", ""), row.get("faculty_research_areas", ""),
                cluster_id=row.get("specialization_cluster"),
            )

            combined_score = round(0.6 * prog["match_score"] + 0.4 * fit_score, 3)

            flat_programs.append({
                "university": prog["university_name"],
                "program_name": prog["program_name"],
                "specialization": specialization,
                "duration": duration_mode["duration"],
                "mode": duration_mode["mode"],
                "mode_is_placeholder": duration_mode["mode_is_placeholder"],
                "specialization_fit_score": fit_score,
                "match_score": combined_score,
                "admit_probability": prog["admit_probability"],
                "tier": prog["tier"],
            })

    flat_programs = sorted(flat_programs, key=lambda p: p["match_score"], reverse=True)

    state["programs"] = flat_programs
    state["status"] = "program_done"
    return state

## 8. Test run

In [15]:
MOCK_STATE = {
    "student_id": "test-001",
    "profile": {
        "gpa": 8.6, "gpa_scale": 10.0,
        "target_program": "Computer Science", "target_degree": "MS",
        "test_scores": {"GRE": {"quant": 165}},
        "interests_text": "distributed systems, cloud infrastructure, and large-scale backend engineering",
    },
    "preferences": {"budget_max_usd": 45000, "priority": "research"},
    "extracurricular": {"profile_strength_score": 0.72},
    "matched_universities": [
        {
            "university": "Stanford University",
            "programs": [
                {"university_name": "Stanford University", "program_name": "MS Computer Science",
                 "match_score": 0.787, "admit_probability": 0.566, "tier": "Target"},
                {"university_name": "Stanford University", "program_name": "MS Computer Science - Systems",
                 "match_score": 0.793, "admit_probability": 0.566, "tier": "Target"},
            ],
        },
    ],
}

result = program_agent(MOCK_STATE)
for p in result["programs"]:
    print(f"{p['university']} -- {p['program_name']}")
    print(f"  specialization={p['specialization']} | duration={p['duration']} | mode={p['mode']}"
          f"{' (placeholder)' if p['mode_is_placeholder'] else ''}")
    print(f"  specialization_fit={p['specialization_fit_score']} | combined match_score={p['match_score']}")
    print()

Stanford University -- MS Computer Science - Systems
  specialization=Systems | duration=24 months | mode=On-campus (placeholder)
  specialization_fit=0.378 | combined match_score=0.627

Stanford University -- MS Computer Science
  specialization=Computer Science | duration=30 months | mode=On-campus (placeholder)
  specialization_fit=0.384 | combined match_score=0.626



In [19]:
evaluation_cases = [
    {
        "ranked_programs": [
            "MS Computer Science - Artificial Intelligence",
            "MS Data Science",
            "MS Computer Science - Systems"
        ],
        "relevant_programs": [
            "MS Computer Science - Artificial Intelligence",
            "MS Data Science"
        ]
    }
]

In [20]:
def top_k_accuracy(ranked_programs, relevant_programs, k=3):
    top_k = [p.lower() for p in ranked_programs[:k]]
    relevant = {p.lower() for p in relevant_programs}
    return int(any(p in relevant for p in top_k))


correct_top1 = 0
correct_top3 = 0

for case in evaluation_cases:
    ranked = case["ranked_programs"]
    relevant = case["relevant_programs"]

    correct_top1 += top_k_accuracy(ranked, relevant, k=1)
    correct_top3 += top_k_accuracy(ranked, relevant, k=3)

n = len(evaluation_cases)

print(f"Top-1 Accuracy: {correct_top1 / n:.2%}")
print(f"Top-3 Accuracy: {correct_top3 / n:.2%}")

Top-1 Accuracy: 100.00%
Top-3 Accuracy: 100.00%


In [21]:
top1_accuracy = correct_top1 / len(evaluation_cases)
top3_accuracy = correct_top3 / len(evaluation_cases)

print(f"Top-1 Accuracy: {top1_accuracy:.2%}")
print(f"Top-3 Accuracy: {top3_accuracy:.2%}")

Top-1 Accuracy: 100.00%
Top-3 Accuracy: 100.00%


## 9. Feeding into Agent 4 and Agent 6
`state["programs"]` (flat, ranked list) is what Agent 4 (Financial) scopes cost estimates against, and what Agent 6 (Career) evaluates research/job alignment against -- both read this list directly rather than re-deriving it from Agent 2's university-grouped output.

In [16]:
top_program_for_agent4 = result["programs"][0]
print("Top program passed to Agent 4:", top_program_for_agent4["university"], "-", top_program_for_agent4["program_name"])

Top program passed to Agent 4: Stanford University - MS Computer Science - Systems
